# 📘 Testing Data Pipelines (pytest Patterns)
## Databricks Notebook — Quality Assurance for DE

**What you'll learn:**
- Unit testing pipeline functions (assert patterns)
- Testing data transformations (input → expected output)
- Data quality test suites (completeness, uniqueness, referential integrity)
- Testing patterns: Arrange-Act-Assert, parameterized, fixtures
- Integration testing (end-to-end pipeline verification)
- Test data factories, property-based testing concepts
- CI/CD for data pipelines

**Note:** pytest doesn't run directly in notebooks. We use equivalent assert-based patterns
with a test runner wrapper. Equivalent pytest code shown for CI/CD.

**Prerequisites:** Notebooks 01-18

---

---
# 📗 Section 1: Why Testing Matters for DE

**Without tests:** "We deployed a change and broke 3 downstream dashboards. Found out 2 days later."

**With tests:** "The test suite caught the schema change before deployment. Zero production impact."

**Types of tests for DE:**
| Type | What it Tests | Speed | Example |
|---|---|---|---|
| Unit | Single function | ⚡ Fast | Does `clean_email()` lowercase correctly? |
| Integration | End-to-end flow | 🔄 Medium | Does Bronze → Silver produce expected output? |
| Data Quality | Data correctness | 🔄 Medium | Are there NULL primary keys? |
| Regression | No unintended changes | 🐌 Slow | Did output change from last known-good run? |

---
# 📗 Section 2: Test Runner & Unit Testing

In [ ]:
# Test runner for notebook environment
import traceback
from datetime import datetime

class TestRunner:
    """Simple test runner for Databricks notebooks."""
    
    def __init__(self, suite_name="Test Suite"):
        self.suite_name = suite_name
        self.results = []
    
    def run_test(self, test_func):
        """Run a single test function and record result."""
        name = test_func.__name__
        try:
            test_func()
            self.results.append({"name": name, "status": "PASS", "error": None})
        except AssertionError as e:
            self.results.append({"name": name, "status": "FAIL", "error": str(e)})
        except Exception as e:
            self.results.append({"name": name, "status": "ERROR", "error": f"{type(e).__name__}: {e}"})
    
    def run_all(self, tests):
        """Run all test functions."""
        for test in tests:
            self.run_test(test)
        self.report()
    
    def report(self):
        """Print test results."""
        passed = sum(1 for r in self.results if r["status"] == "PASS")
        failed = sum(1 for r in self.results if r["status"] == "FAIL")
        errors = sum(1 for r in self.results if r["status"] == "ERROR")
        
        print(f"\n{'='*60}")
        print(f"TEST RESULTS: {self.suite_name}")
        print(f"{'='*60}")
        for r in self.results:
            icon = {"PASS": "✅", "FAIL": "❌", "ERROR": "💥"}[r["status"]]
            print(f"  {icon} {r['name']}")
            if r["error"]:
                print(f"      → {r['error']}")
        print(f"{'='*60}")
        print(f"  Total: {len(self.results)} | Passed: {passed} | Failed: {failed} | Errors: {errors}")
        print(f"{'='*60}")

print("✅ TestRunner ready")

In [ ]:
# Unit tests for pipeline functions
def clean_email(email):
    """Clean an email address."""
    if email is None:
        return None
    return email.strip().lower()

def classify_order(amount):
    """Classify order by amount."""
    if amount is None or amount < 0:
        return "invalid"
    if amount >= 4000:
        return "premium"
    if amount >= 2000:
        return "standard"
    return "basic"

# Write tests
def test_clean_email_normal():
    assert clean_email("  John@Gmail.COM  ") == "john@gmail.com"

def test_clean_email_none():
    assert clean_email(None) is None

def test_clean_email_already_clean():
    assert clean_email("test@example.com") == "test@example.com"

def test_classify_premium():
    assert classify_order(5000) == "premium"

def test_classify_standard():
    assert classify_order(2500) == "standard"

def test_classify_basic():
    assert classify_order(500) == "basic"

def test_classify_invalid_none():
    assert classify_order(None) == "invalid"

def test_classify_invalid_negative():
    assert classify_order(-100) == "invalid"

# Run tests
runner = TestRunner("Unit Tests: Pipeline Functions")
runner.run_all([
    test_clean_email_normal, test_clean_email_none, test_clean_email_already_clean,
    test_classify_premium, test_classify_standard, test_classify_basic,
    test_classify_invalid_none, test_classify_invalid_negative
])

---
# 📗 Section 3: Testing Data Transformations

In [ ]:
from pyspark.sql.functions import *

# Function under test: deduplication
def deduplicate_orders(df):
    """Deduplicate orders keeping the latest by created_at."""
    from pyspark.sql.window import Window
    w = Window.partitionBy("order_id").orderBy(col("created_at").desc())
    return df.withColumn("_rn", row_number().over(w)).filter("_rn = 1").drop("_rn")

# Tests for the deduplication function
def test_dedup_removes_duplicates():
    """Dedup should remove duplicate order_ids."""
    test_data = [(1, 100.0, "2024-01-01"), (1, 100.0, "2024-01-02"), (2, 200.0, "2024-01-01")]
    df = spark.createDataFrame(test_data, ["order_id", "total_amount", "created_at"])
    result = deduplicate_orders(df)
    assert result.count() == 2, f"Expected 2 rows, got {result.count()}"

def test_dedup_keeps_latest():
    """Dedup should keep the row with latest created_at."""
    test_data = [(1, 100.0, "2024-01-01"), (1, 150.0, "2024-01-05")]
    df = spark.createDataFrame(test_data, ["order_id", "total_amount", "created_at"])
    result = deduplicate_orders(df)
    amount = result.collect()[0].total_amount
    assert amount == 150.0, f"Expected 150.0 (latest), got {amount}"

def test_dedup_no_duplicates():
    """Dedup on data without duplicates should return same count."""
    test_data = [(1, 100.0, "2024-01-01"), (2, 200.0, "2024-01-02"), (3, 300.0, "2024-01-03")]
    df = spark.createDataFrame(test_data, ["order_id", "total_amount", "created_at"])
    result = deduplicate_orders(df)
    assert result.count() == 3, f"Expected 3 rows, got {result.count()}"

runner = TestRunner("Transformation Tests: Deduplication")
runner.run_all([test_dedup_removes_duplicates, test_dedup_keeps_latest, test_dedup_no_duplicates])

---
# 📗 Section 4: Data Quality Tests

In [ ]:
# Data quality test suite for any table
from pyspark.sql.functions import *

def test_no_null_pks():
    """Primary key (order_id) should never be NULL."""
    df = spark.table("techmart_dw.orders")
    null_count = df.filter("order_id IS NULL").count()
    assert null_count == 0, f"Found {null_count} NULL primary keys"

def test_unique_pks():
    """Primary key should be unique."""
    df = spark.table("techmart_dw.orders")
    total = df.count()
    distinct = df.select("order_id").distinct().count()
    assert total == distinct, f"Duplicates found: {total} total vs {distinct} distinct"

def test_positive_amounts():
    """total_amount should be non-negative."""
    df = spark.table("techmart_dw.orders")
    negative = df.filter("total_amount < 0").count()
    assert negative == 0, f"Found {negative} negative amounts"

def test_valid_status():
    """Status should be one of the expected values."""
    valid_statuses = {"completed", "shipped", "processing", "pending", "cancelled", "returned"}
    df = spark.table("techmart_dw.orders")
    actual_statuses = set(row.status for row in df.select("status").distinct().collect())
    invalid = actual_statuses - valid_statuses
    assert len(invalid) == 0, f"Invalid statuses found: {invalid}"

def test_referential_integrity():
    """All order customer_ids should exist in customers table."""
    orders = spark.table("techmart_dw.orders")
    customers = spark.table("techmart_dw.customers")
    orphans = orders.join(customers, "customer_id", "left_anti").count()
    assert orphans == 0, f"Found {orphans} orders with no matching customer"

def test_data_freshness():
    """Most recent order should be within last 2 years (data isn't stale)."""
    df = spark.table("techmart_dw.orders")
    max_date = df.agg(max("order_date")).collect()[0][0]
    assert max_date is not None, "No order dates found"

runner = TestRunner("Data Quality: techmart_dw.orders")
runner.run_all([
    test_no_null_pks, test_unique_pks, test_positive_amounts,
    test_valid_status, test_referential_integrity, test_data_freshness
])

In [ ]:
# ============================================
# 🎯 YOUR TURN: Write Quality Tests
# ============================================
# Write data quality tests for techmart_dw.customers:
# 1. test_customer_id_not_null — customer_id should never be NULL
# 2. test_customer_id_unique — no duplicate customer_ids
# 3. test_email_format — emails (non-null) should contain '@'
# 4. test_segment_valid — customer_segment in ('Premium','Standard','Basic','New')
# 5. test_lifetime_value_positive — lifetime_value >= 0
#
# Run with TestRunner
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
from pyspark.sql.functions import *

def test_customer_id_not_null():
    df = spark.table("techmart_dw.customers")
    nulls = df.filter("customer_id IS NULL").count()
    assert nulls == 0, f"Found {nulls} NULL customer_ids"

def test_customer_id_unique():
    df = spark.table("techmart_dw.customers")
    total = df.count()
    distinct = df.select("customer_id").distinct().count()
    assert total == distinct, f"Duplicates: {total - distinct}"

def test_email_format():
    df = spark.table("techmart_dw.customers").filter("email IS NOT NULL")
    bad_emails = df.filter(~col("email").contains("@")).count()
    assert bad_emails == 0, f"Found {bad_emails} emails without @"

def test_segment_valid():
    valid = {"Premium", "Standard", "Basic", "New"}
    df = spark.table("techmart_dw.customers")
    actual = set(r.customer_segment for r in df.select("customer_segment").distinct().collect())
    invalid = actual - valid
    assert len(invalid) == 0, f"Invalid segments: {invalid}"

def test_lifetime_value_positive():
    df = spark.table("techmart_dw.customers")
    negative = df.filter("lifetime_value < 0").count()
    assert negative == 0, f"Found {negative} negative lifetime values"

runner = TestRunner("Data Quality: customers")
runner.run_all([test_customer_id_not_null, test_customer_id_unique, test_email_format, test_segment_valid, test_lifetime_value_positive])

---
# 📗 Section 5: Testing Patterns

## Parameterized Tests

In [ ]:
# Parameterized: same test logic, different inputs
def run_parameterized_quality_checks(tables_config):
    """Run standard quality checks across multiple tables."""
    runner = TestRunner("Parameterized Quality Checks")
    
    for table, pk_col in tables_config:
        # Generate test functions dynamically
        def make_null_test(t, pk):
            def test():
                df = spark.table(t)
                nulls = df.filter(f"{pk} IS NULL").count()
                assert nulls == 0, f"{t}.{pk} has {nulls} NULLs"
            test.__name__ = f"test_{t.split('.')[-1]}_{pk}_not_null"
            return test
        
        def make_count_test(t):
            def test():
                count = spark.table(t).count()
                assert count > 0, f"{t} is empty!"
            test.__name__ = f"test_{t.split('.')[-1]}_not_empty"
            return test
        
        runner.run_test(make_null_test(table, pk_col))
        runner.run_test(make_count_test(table))
    
    runner.report()

# Run across multiple tables
run_parameterized_quality_checks([
    ("techmart_dw.orders", "order_id"),
    ("techmart_dw.customers", "customer_id"),
    ("techmart_dw.products", "product_id"),
    ("techmart_dw.payments", "payment_id"),
])

---
# 📗 Section 6: Integration Testing

In [ ]:
# Integration test: verify full pipeline output
from pyspark.sql.functions import *

def test_silver_orders_pipeline():
    """Integration test: Bronze → Silver orders pipeline."""
    # Arrange: read bronze
    bronze = spark.table("techmart_dw.orders")
    initial_count = bronze.count()
    
    # Act: run silver transformation
    silver = (bronze
        .filter("order_id IS NOT NULL AND customer_id IS NOT NULL")
        .filter("total_amount >= 0")
        .withColumn("status", lower(trim(col("status"))))
        .dropDuplicates(["order_id"])
    )
    
    # Assert: verify properties
    silver_count = silver.count()
    assert silver_count > 0, "Silver table is empty"
    assert silver_count <= initial_count, f"Silver ({silver_count}) > Bronze ({initial_count})"
    
    # Assert: no nulls in required fields
    null_ids = silver.filter("order_id IS NULL").count()
    assert null_ids == 0, f"Silver has {null_ids} NULL order_ids"
    
    # Assert: status is lowercase
    upper_status = silver.filter(col("status").rlike("[A-Z]")).count()
    assert upper_status == 0, f"Found {upper_status} non-lowercase statuses"
    
    # Assert: no negative amounts
    neg = silver.filter("total_amount < 0").count()
    assert neg == 0, f"Found {neg} negative amounts in silver"

runner = TestRunner("Integration: Silver Orders Pipeline")
runner.run_all([test_silver_orders_pipeline])

---
# 📗 Section 7: Test Data Factories

In [ ]:
import random
from datetime import date, timedelta

def generate_test_orders(n=100, null_rate=0.0, invalid_rate=0.0, seed=42):
    """Generate test order data with configurable properties.
    
    Args:
        n: number of records
        null_rate: fraction of records with NULL customer_id
        invalid_rate: fraction with negative amounts
        seed: random seed for reproducibility
    """
    random.seed(seed)
    records = []
    
    for i in range(1, n + 1):
        customer_id = None if random.random() < null_rate else random.randint(1, 1000)
        amount = -random.uniform(1, 100) if random.random() < invalid_rate else random.uniform(10, 5000)
        
        records.append((
            i,  # order_id
            customer_id,
            (date(2024, 1, 1) + timedelta(days=random.randint(0, 180))).isoformat(),
            round(amount, 2),
            random.choice(["completed", "pending", "cancelled"])
        ))
    
    return spark.createDataFrame(records, ["order_id", "customer_id", "order_date", "total_amount", "status"])

# Generate different test scenarios
clean_data = generate_test_orders(100, null_rate=0.0, invalid_rate=0.0)
messy_data = generate_test_orders(100, null_rate=0.1, invalid_rate=0.05)

print(f"Clean data: {clean_data.count()} rows, {clean_data.filter('customer_id IS NULL').count()} nulls")
print(f"Messy data: {messy_data.count()} rows, {messy_data.filter('customer_id IS NULL').count()} nulls, {messy_data.filter('total_amount < 0').count()} negative")

---
# 🚀 Section 8: Mini Projects

## Project 1: Complete Pipeline Test Suite

In [ ]:
# Complete test suite for a pipeline
from pyspark.sql.functions import *

class PipelineTestSuite:
    """Comprehensive test suite for data pipelines."""
    
    def __init__(self, name):
        self.name = name
        self.runner = TestRunner(name)
    
    def test_schema(self, df, expected_columns):
        """Verify DataFrame has expected columns."""
        def test():
            actual = set(df.columns)
            missing = set(expected_columns) - actual
            assert len(missing) == 0, f"Missing columns: {missing}"
        test.__name__ = "test_schema_columns"
        self.runner.run_test(test)
    
    def test_row_count(self, df, min_rows, max_rows=None):
        """Verify row count is within expected range."""
        def test():
            count = df.count()
            assert count >= min_rows, f"Too few rows: {count} < {min_rows}"
            if max_rows:
                assert count <= max_rows, f"Too many rows: {count} > {max_rows}"
        test.__name__ = "test_row_count"
        self.runner.run_test(test)
    
    def test_no_nulls(self, df, columns):
        """Verify no NULLs in specified columns."""
        for col_name in columns:
            def make_test(c):
                def test():
                    nulls = df.filter(f"{c} IS NULL").count()
                    assert nulls == 0, f"{c} has {nulls} NULLs"
                test.__name__ = f"test_no_null_{c}"
                return test
            self.runner.run_test(make_test(col_name))
    
    def test_unique(self, df, columns):
        """Verify uniqueness of column combination."""
        def test():
            total = df.count()
            distinct = df.select(columns).distinct().count()
            assert total == distinct, f"Duplicates: {total - distinct}"
        test.__name__ = f"test_unique_{'_'.join(columns)}"
        self.runner.run_test(test)
    
    def report(self):
        self.runner.report()

# Run comprehensive test suite
orders = spark.table("techmart_dw.orders")
suite = PipelineTestSuite("Orders Pipeline Test Suite")
suite.test_schema(orders, ["order_id", "customer_id", "total_amount", "status", "order_date"])
suite.test_row_count(orders, min_rows=10000)
suite.test_no_nulls(orders, ["order_id", "customer_id"])
suite.test_unique(orders, ["order_id"])
suite.report()

---
# 🏆 Section 9: Interview Questions

## Q1: How do you test data pipelines?

1. **Unit tests:** Test each transformation function in isolation with known inputs
2. **Integration tests:** Run full pipeline on sample data, verify output properties
3. **Data quality tests:** Check completeness, uniqueness, referential integrity
4. **Regression tests:** Compare output to known-good baseline
5. **Run in CI/CD:** Automated on every code change

## Q2: Unit vs integration tests for DE?

- **Unit:** Test one function (e.g., `clean_email("  X@Y.COM  ")` → `"x@y.com"`)
- **Integration:** Test full pipeline flow (Bronze → Silver → Gold), verify final state
- **Unit tests are fast** (milliseconds), integration tests are slower (seconds/minutes)
- **Both are needed:** Unit catches logic bugs, integration catches wiring bugs

## Q3: Ensure data quality in production?

1. Quality tests run after every pipeline execution
2. Automated alerts on test failures
3. Quarantine pattern: bad data goes to separate table
4. Quality metrics tracked over time (trending)
5. SLA monitoring (freshness, completeness thresholds)

## Q4: Testing strategy for a new pipeline?

1. Start with data quality tests on the source (understand the data)
2. Write unit tests for each transformation function
3. Build integration test with sample data
4. Add regression test (capture golden output)
5. Set up CI/CD to run tests on every PR
6. Monitor quality metrics in production

## Q5: Testing MERGE operations?

1. Create test data with known inserts, updates, and deletes
2. Run MERGE
3. Verify: new records inserted (count increased)
4. Verify: existing records updated (values changed)
5. Verify: deleted records handled (soft delete or removed)
6. Verify: unchanged records untouched

## Q6: Property-based testing?

Test **properties** (invariants) rather than specific examples:
- "Output count <= input count" (for filters)
- "No NULLs in output required columns" (for cleaners)
- "Running twice gives same result" (idempotency)
The testing framework generates random inputs to find edge cases you didn't think of.

## Q7: Test data management?

- **Factories:** Generate controlled test data with known properties
- **Fixtures:** Reusable setup/teardown for test databases
- **Sampling:** Use a representative sample of production data (anonymized)
- **Seeded random:** Reproducible test data with `random.seed(42)`

## Q8: Test data freshness?

```python
def test_freshness():
    max_date = spark.table("orders").agg(max("created_at")).collect()[0][0]
    hours_old = (datetime.now() - max_date).total_seconds() / 3600
    assert hours_old < 24, f"Data is {hours_old:.1f} hours old (SLA: 24h)"
```

---
# 📗 Section 4: Fixtures, Parametrize & Mocking

These are the three most important pytest features for DE testing:
- **Fixtures**: Reusable test setup (database connections, sample data)
- **Parametrize**: Run the same test with multiple inputs
- **Mocking**: Replace external dependencies (APIs, databases) with fakes

In [ ]:
import pytest
import json
import os
import tempfile
from unittest.mock import Mock, patch, MagicMock, call
from typing import List, Dict, Any

# ============================================================
# FIXTURES: Reusable test setup
# ============================================================

# Scope controls how often the fixture is created:
# "function" (default): new instance per test
# "class": shared within a test class
# "module": shared within a module
# "session": shared across entire test run

@pytest.fixture(scope="function")
def sample_orders():
    """Fresh order data for each test."""
    return [
        {"order_id": 1, "customer_id": 101, "amount": 150.0, "status": "completed"},
        {"order_id": 2, "customer_id": 102, "amount": 200.0, "status": "pending"},
        {"order_id": 3, "customer_id": 101, "amount": 75.0,  "status": "cancelled"},
        {"order_id": 4, "customer_id": 103, "amount": 300.0, "status": "completed"},
        {"order_id": 5, "customer_id": 102, "amount": 50.0,  "status": "completed"},
    ]

@pytest.fixture(scope="module")
def temp_dir():
    """Temporary directory shared across all tests in the module."""
    with tempfile.TemporaryDirectory() as tmpdir:
        yield tmpdir  # yield instead of return for cleanup

@pytest.fixture
def mock_api_client():
    """Mock API client that returns predictable data."""
    client = Mock()
    client.get_orders.return_value = [
        {"id": 1, "amount": 100.0},
        {"id": 2, "amount": 200.0},
    ]
    client.get_customer.return_value = {"id": 101, "name": "Alice", "tier": "gold"}
    client.post_order.return_value = {"id": 99, "status": "created"}
    return client

@pytest.fixture
def sample_csv(temp_dir):
    """Create a sample CSV file for testing."""
    csv_path = os.path.join(temp_dir, "orders.csv")
    with open(csv_path, "w") as f:
        f.write("order_id,amount,status\n")
        f.write("1,100.0,completed\n")
        f.write("2,200.0,pending\n")
        f.write("3,bad_amount,shipped\n")  # Bad data
    return csv_path

print("Fixtures defined: sample_orders, temp_dir, mock_api_client, sample_csv")
print("These would be used in actual pytest test files.")

In [ ]:
# ============================================================
# PARAMETRIZE: Test multiple inputs with one test function
# ============================================================

# The functions we want to test
def validate_order(order: dict) -> tuple:
    """Returns (is_valid, error_message)."""
    if not order.get("order_id"):
        return False, "Missing order_id"
    if order.get("amount", 0) <= 0:
        return False, f"Invalid amount: {order.get('amount')}"
    valid_statuses = {"pending", "processing", "shipped", "delivered", "cancelled"}
    if order.get("status") not in valid_statuses:
        return False, f"Invalid status: {order.get('status')}"
    return True, None

def calculate_tax(amount: float, rate: float = 0.08) -> float:
    """Calculate tax amount."""
    if amount < 0:
        raise ValueError("Amount cannot be negative")
    return round(amount * rate, 2)

def normalize_status(status: str) -> str:
    """Normalize status string."""
    return status.strip().lower()

# Parametrized test cases (what pytest would run)
validate_test_cases = [
    # (order, expected_valid, expected_error_contains)
    ({"order_id": 1, "amount": 100.0, "status": "pending"}, True, None),
    ({"order_id": None, "amount": 100.0, "status": "pending"}, False, "order_id"),
    ({"order_id": 1, "amount": -50.0, "status": "pending"}, False, "amount"),
    ({"order_id": 1, "amount": 0, "status": "pending"}, False, "amount"),
    ({"order_id": 1, "amount": 100.0, "status": "unknown"}, False, "status"),
    ({"order_id": 1, "amount": 100.0, "status": "COMPLETED"}, False, "status"),
]

print("Parametrized validation tests:")
all_passed = True
for order, expected_valid, expected_error in validate_test_cases:
    is_valid, error = validate_order(order)
    
    passed = is_valid == expected_valid
    if expected_error and error:
        passed = passed and expected_error in error
    
    status = "✅" if passed else "❌"
    print(f"  {status} order_id={order.get('order_id')}, amount={order.get('amount')}, "
          f"status={order.get('status')} → valid={is_valid}")
    if not passed:
        all_passed = False

print(f"\n{'All tests passed!' if all_passed else 'Some tests failed!'}")

In [ ]:
# ============================================================
# MOCKING: Replace external dependencies
# ============================================================

# The pipeline function we want to test
def ingest_orders_from_api(api_client, target_db) -> dict:
    """
    Ingest orders from API and save to database.
    This function has external dependencies we need to mock.
    """
    # Fetch from API
    orders = api_client.get_orders(status="pending")
    
    if not orders:
        return {"status": "no_data", "count": 0}
    
    # Transform
    processed = []
    for order in orders:
        processed.append({
            "id": order["id"],
            "amount": order["amount"],
            "tax": round(order["amount"] * 0.08, 2),
            "total": round(order["amount"] * 1.08, 2),
        })
    
    # Save to DB
    target_db.bulk_insert("orders", processed)
    
    return {"status": "success", "count": len(processed)}

# Test with mocks (simulating what pytest would do)
def test_ingest_orders_success():
    # Arrange: create mocks
    mock_api = Mock()
    mock_api.get_orders.return_value = [
        {"id": 1, "amount": 100.0},
        {"id": 2, "amount": 200.0},
    ]
    
    mock_db = Mock()
    mock_db.bulk_insert.return_value = None
    
    # Act
    result = ingest_orders_from_api(mock_api, mock_db)
    
    # Assert
    assert result["status"] == "success"
    assert result["count"] == 2
    
    # Verify API was called correctly
    mock_api.get_orders.assert_called_once_with(status="pending")
    
    # Verify DB was called with transformed data
    mock_db.bulk_insert.assert_called_once()
    call_args = mock_db.bulk_insert.call_args
    assert call_args[0][0] == "orders"  # Table name
    records = call_args[0][1]
    assert len(records) == 2
    assert records[0]["tax"] == 8.0   # 100 * 0.08
    assert records[0]["total"] == 108.0  # 100 * 1.08
    
    print("✅ test_ingest_orders_success passed")

def test_ingest_orders_empty():
    mock_api = Mock()
    mock_api.get_orders.return_value = []
    mock_db = Mock()
    
    result = ingest_orders_from_api(mock_api, mock_db)
    
    assert result["status"] == "no_data"
    assert result["count"] == 0
    mock_db.bulk_insert.assert_not_called()  # DB should NOT be called
    
    print("✅ test_ingest_orders_empty passed")

def test_ingest_orders_api_failure():
    mock_api = Mock()
    mock_api.get_orders.side_effect = ConnectionError("API unavailable")
    mock_db = Mock()
    
    try:
        ingest_orders_from_api(mock_api, mock_db)
        print("❌ Should have raised ConnectionError")
    except ConnectionError:
        print("✅ test_ingest_orders_api_failure passed (exception propagated correctly)")

test_ingest_orders_success()
test_ingest_orders_empty()
test_ingest_orders_api_failure()

## 4.2 Testing Data Quality Functions

In [ ]:
# ============================================================
# DATA QUALITY TEST PATTERNS
# ============================================================

# Functions to test
def check_no_nulls(records: list, column: str) -> dict:
    nulls = sum(1 for r in records if r.get(column) is None)
    return {"passed": nulls == 0, "null_count": nulls, "total": len(records)}

def check_unique(records: list, column: str) -> dict:
    values = [r.get(column) for r in records]
    dupes = len(values) - len(set(values))
    return {"passed": dupes == 0, "duplicate_count": dupes}

def check_value_range(records: list, column: str, min_val, max_val) -> dict:
    out_of_range = sum(
        1 for r in records
        if r.get(column) is not None and not (min_val <= r[column] <= max_val)
    )
    return {"passed": out_of_range == 0, "out_of_range_count": out_of_range}

# Test suite
test_data = [
    {"order_id": 1, "amount": 100.0, "status": "completed"},
    {"order_id": 2, "amount": 200.0, "status": "pending"},
    {"order_id": 3, "amount": 75.0,  "status": "completed"},
]

# Run quality checks
checks = [
    ("no_nulls(order_id)", check_no_nulls(test_data, "order_id")),
    ("unique(order_id)", check_unique(test_data, "order_id")),
    ("amount in [0, 10000]", check_value_range(test_data, "amount", 0, 10000)),
]

print("Data Quality Test Results:")
for check_name, result in checks:
    status = "✅ PASS" if result["passed"] else "❌ FAIL"
    print(f"  {status}: {check_name} — {result}")

# Test with bad data
bad_data = [
    {"order_id": 1, "amount": 100.0},
    {"order_id": 1, "amount": -50.0},  # Duplicate ID + negative amount
    {"order_id": None, "amount": 200.0},  # Null ID
]

print("\nBad data quality checks:")
for check_name, result in [
    ("no_nulls(order_id)", check_no_nulls(bad_data, "order_id")),
    ("unique(order_id)", check_unique(bad_data, "order_id")),
    ("amount in [0, 10000]", check_value_range(bad_data, "amount", 0, 10000)),
]:
    status = "✅ PASS" if result["passed"] else "❌ FAIL"
    print(f"  {status}: {check_name} — {result}")

---
# 📗 Section 5: Practice Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Write a Complete Test Suite
# ============================================================

# The pipeline to test
def process_sales_data(records: list) -> dict:
    """
    Process sales records:
    1. Filter out cancelled orders
    2. Calculate tax (8%)
    3. Group by customer
    4. Return summary
    """
    if not records:
        return {"customers": {}, "total_revenue": 0, "order_count": 0}
    
    active = [r for r in records if r.get("status") != "cancelled"]
    
    customers = {}
    for record in active:
        cid = record["customer_id"]
        amount = record["amount"]
        tax = round(amount * 0.08, 2)
        
        if cid not in customers:
            customers[cid] = {"orders": 0, "revenue": 0, "tax": 0}
        
        customers[cid]["orders"] += 1
        customers[cid]["revenue"] += amount
        customers[cid]["tax"] += tax
    
    total_revenue = sum(c["revenue"] for c in customers.values())
    
    return {
        "customers": customers,
        "total_revenue": round(total_revenue, 2),
        "order_count": len(active),
    }

# Test suite
def run_test_suite():
    tests_passed = 0
    tests_failed = 0
    
    def assert_test(name, condition, detail=""):
        nonlocal tests_passed, tests_failed
        if condition:
            print(f"  ✅ {name}")
            tests_passed += 1
        else:
            print(f"  ❌ {name}: {detail}")
            tests_failed += 1
    
    # Test 1: Empty input
    result = process_sales_data([])
    assert_test("empty_input_returns_zero", result["total_revenue"] == 0)
    assert_test("empty_input_no_customers", result["customers"] == {})
    
    # Test 2: Normal processing
    records = [
        {"customer_id": 1, "amount": 100.0, "status": "completed"},
        {"customer_id": 1, "amount": 200.0, "status": "shipped"},
        {"customer_id": 2, "amount": 150.0, "status": "completed"},
        {"customer_id": 1, "amount": 50.0,  "status": "cancelled"},  # Should be excluded
    ]
    result = process_sales_data(records)
    assert_test("cancelled_excluded", result["order_count"] == 3)
    assert_test("correct_total", result["total_revenue"] == 450.0)
    assert_test("customer_1_orders", result["customers"][1]["orders"] == 2)
    assert_test("customer_1_revenue", result["customers"][1]["revenue"] == 300.0)
    assert_test("customer_2_exists", 2 in result["customers"])
    
    # Test 3: Tax calculation
    records = [{"customer_id": 1, "amount": 100.0, "status": "completed"}]
    result = process_sales_data(records)
    assert_test("tax_calculated", result["customers"][1]["tax"] == 8.0)
    
    print(f"\n{'='*40}")
    print(f"Results: {tests_passed} passed, {tests_failed} failed")
    return tests_failed == 0

success = run_test_suite()

---
# 📗 Section 5: Advanced Pytest Patterns for DE

## Fixtures -- Reusable Test Setup

```python
# conftest.py -- shared fixtures across all tests
import pytest
from pyspark.sql import SparkSession

@pytest.fixture(scope="session")
def spark():
    return (SparkSession.builder
        .master("local[2]")
        .appName("test")
        .config("spark.sql.shuffle.partitions", "2")
        .getOrCreate())

@pytest.fixture
def sample_orders(spark):
    return spark.createDataFrame([
        (1, 42, 100.0, "completed"),
        (2, 17, 200.0, "pending"),
        (3, 55, 300.0, "shipped"),
    ], ["order_id", "customer_id", "amount", "status"])

@pytest.fixture
def empty_df(spark):
    from pyspark.sql.types import StructType, StructField, IntegerType, StringType
    schema = StructType([
        StructField("order_id", IntegerType()),
        StructField("status", StringType()),
    ])
    return spark.createDataFrame([], schema)
```

## Parametrize -- Test Multiple Cases

```python
@pytest.mark.parametrize("status,expected_count", [
    ("completed", 1),
    ("pending", 1),
    ("shipped", 1),
    ("cancelled", 0),
])
def test_filter_by_status(sample_orders, status, expected_count):
    result = sample_orders.filter(f"status = '{status}'")
    assert result.count() == expected_count
```

## Mocking External Dependencies

```python
from unittest.mock import patch, MagicMock

def test_api_ingestion_with_mock():
    mock_response = MagicMock()
    mock_response.json.return_value = [{"id": 1, "amount": 100}]
    mock_response.status_code = 200
    
    with patch("requests.get", return_value=mock_response):
        result = fetch_orders_from_api("https://api.example.com/orders")
        assert len(result) == 1
        assert result[0]["amount"] == 100
```

## Testing Data Quality

```python
def test_no_null_order_ids(sample_orders):
    null_count = sample_orders.filter("order_id IS NULL").count()
    assert null_count == 0, f"Found {null_count} NULL order_ids"

def test_amounts_are_positive(sample_orders):
    negative = sample_orders.filter("amount <= 0").count()
    assert negative == 0, f"Found {negative} non-positive amounts"

def test_status_values_valid(sample_orders):
    valid_statuses = {"completed", "pending", "shipped", "cancelled"}
    actual_statuses = {row.status for row in sample_orders.select("status").collect()}
    invalid = actual_statuses - valid_statuses
    assert not invalid, f"Invalid statuses found: {invalid}"
```

In [ ]:
# Pytest patterns for data engineering
import unittest
from unittest.mock import patch, MagicMock

print("Pytest Patterns for Data Engineering")
print("=" * 60)

# Simulate the functions we want to test
def clean_orders(records):
    return [r for r in records if r.get("order_id") and r.get("amount", 0) > 0]

def calculate_revenue(records, status="completed"):
    return sum(r["amount"] for r in records if r.get("status") == status)

def fetch_from_api(url):
    import requests
    response = requests.get(url)
    return response.json()

# Test class with fixtures
class TestOrderPipeline(unittest.TestCase):
    
    def setUp(self):
        """Fixture: runs before each test."""
        self.sample_orders = [
            {"order_id": 1, "amount": 100.0, "status": "completed"},
            {"order_id": 2, "amount": 200.0, "status": "pending"},
            {"order_id": None, "amount": 50.0, "status": "completed"},  # NULL
            {"order_id": 4, "amount": -10.0, "status": "completed"},    # Negative
        ]
    
    def test_clean_removes_nulls(self):
        result = clean_orders(self.sample_orders)
        self.assertTrue(all(r["order_id"] is not None for r in result))
    
    def test_clean_removes_negatives(self):
        result = clean_orders(self.sample_orders)
        self.assertTrue(all(r["amount"] > 0 for r in result))
    
    def test_revenue_completed_only(self):
        cleaned = clean_orders(self.sample_orders)
        revenue = calculate_revenue(cleaned, "completed")
        self.assertEqual(revenue, 100.0)  # Only order 1 is valid + completed
    
    def test_revenue_empty_input(self):
        """Edge case: empty input."""
        self.assertEqual(calculate_revenue([]), 0.0)
    
    @patch("requests.get")
    def test_api_fetch_with_mock(self, mock_get):
        """Test API call without making real HTTP request."""
        mock_response = MagicMock()
        mock_response.json.return_value = [{"id": 1, "amount": 100}]
        mock_get.return_value = mock_response
        
        result = fetch_from_api("https://api.example.com/orders")
        self.assertEqual(len(result), 1)
        mock_get.assert_called_once_with("https://api.example.com/orders")


# Run tests
print("\nRunning tests...")
loader = unittest.TestLoader()
suite = loader.loadTestsFromTestCase(TestOrderPipeline)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print(f"\n{'All tests passed!' if result.wasSuccessful() else 'Some tests failed!'}")


---
# ✅ Notebook Complete!

**What was covered:**
- TestRunner class for notebook-based testing
- Unit tests: testing transformation functions with assert
- Data transformation tests: deduplication verification
- Data quality tests: completeness, uniqueness, referential integrity, freshness
- Parameterized tests: same checks across multiple tables
- Integration tests: full pipeline verification
- Test data factories: generate controlled test data
- PipelineTestSuite class: reusable comprehensive testing
- 8 interview questions

**Next:** Notebook 20 — Pathlib & OS for Data Engineering

---